# ❤️ RHD Heart Sound Classification Model

**Medical Triage Tool for Resource-Limited Settings**

MobileNetV2-based Classification of Heart Sound Spectrograms

---
**References:**
- PhysioNet 2016: Environmental noise robustness
- CirCOR Digiscope 2022: Clinical validation and pediatric data

## 📦 Import Libraries

In [ ]:
import os
import sys
import json
import pickle
import argparse
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from sklearn.utils import class_weight
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version: {np.__version__}")

## ⚙️ Configuration

In [ ]:
class Config:
    """Configuration for the RHD classification model."""
    
    # Data paths - UPDATE THESE
    DATA_PATH = os.environ.get('RHD_DATA_PATH', '../data/processed/heart_grades_classified')
    OUTPUT_PATH = os.environ.get('RHD_OUTPUT_PATH', '../exported_models')
    
    # Model architecture
    IMG_SIZE = (224, 224)
    BATCH_SIZE = 32
    NUM_CLASSES = 4
    CLASS_NAMES = ['Grade_0', 'Grade_1', 'Grade_2', 'Grade_3']
    
    # Training parameters
    EPOCHS = 50
    INITIAL_LR = 0.001
    VALIDATION_SPLIT = 0.3
    TRAIN_SPLIT = 0.7
    RANDOM_SEED = 42
    USE_BALANCED_WEIGHTS = True
    
    # Augmentation
    AUGMENTATION = {
        'flip': True,
        'rotation': 0.15,
        'zoom': 0.15,
        'contrast': 0.1,
        'brightness': 0.1,
    }
    
    # Export
    EXPORT_FORMATS = ['keras', 'tflite', 'saved_model']
    SAVE_HISTORY = True
    SAVE_PLOTS = True
    
    def __init__(self):
        os.makedirs(self.OUTPUT_PATH, exist_ok=True)
        os.makedirs(os.path.join(self.OUTPUT_PATH, 'logs'), exist_ok=True)
        os.makedirs(os.path.join(self.OUTPUT_PATH, 'plots'), exist_ok=True)
        os.makedirs(os.path.join(self.OUTPUT_PATH, 'models'), exist_ok=True)
        self.TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
        self.RUN_NAME = f"rhd_mobilenet_{self.TIMESTAMP}"

# Initialize configuration
config = Config()
print(f"Data path: {config.DATA_PATH}")
print(f"Output path: {config.OUTPUT_PATH}")
print(f"Run name: {config.RUN_NAME}")

## 📂 Data Loading

In [ ]:
class DataLoader:
    """Handles loading and preprocessing of heart sound spectrogram data."""
    
    def __init__(self, config):
        self.config = config
        self.class_names = None
        self.num_classes = None
        self.train_ds = None
        self.val_ds = None
        self.test_ds = None
        self.class_weights = None
        
    def load_data(self):
        """Load the spectrogram dataset from directory structure."""
        print("\n" + "="*60)
        print("📂 LOADING SPECTROGRAM DATASET")
        print("="*60)
        print(f"Data path: {self.config.DATA_PATH}")
        print(f"Image size: {self.config.IMG_SIZE}")
        print(f"Batch size: {self.config.BATCH_SIZE}")
        
        # Load full dataset
        full_ds = tf.keras.preprocessing.image_dataset_from_directory(
            self.config.DATA_PATH,
            validation_split=self.config.VALIDATION_SPLIT,
            subset="training",
            seed=self.config.RANDOM_SEED,
            image_size=self.config.IMG_SIZE,
            batch_size=self.config.BATCH_SIZE,
            label_mode='int',
            color_mode='rgb',
            shuffle=True
        )
        
        self.class_names = full_ds.class_names
        self.num_classes = len(self.class_names)
        print(f"\n📊 Classes found: {self.class_names}")
        print(f"   Number of classes: {self.num_classes}")
        
        # Get validation portion
        val_full_ds = tf.keras.preprocessing.image_dataset_from_directory(
            self.config.DATA_PATH,
            validation_split=self.config.VALIDATION_SPLIT,
            subset="validation",
            seed=self.config.RANDOM_SEED,
            image_size=self.config.IMG_SIZE,
            batch_size=self.config.BATCH_SIZE,
            label_mode='int',
            color_mode='rgb',
            shuffle=False
        )
        
        # Split validation into val and test
        val_full_size = tf.data.experimental.cardinality(val_full_ds).numpy()
        val_size = val_full_size // 2
        test_size = val_full_size - val_size
        
        self.val_ds = val_full_ds.take(val_size)
        self.test_ds = val_full_ds.skip(val_size)
        self.train_ds = full_ds
        
        print(f"\n📊 Dataset splits:")
        print(f"   Training: {tf.data.experimental.cardinality(self.train_ds).numpy()} batches")
        print(f"   Validation: {val_size} batches")
        print(f"   Test: {test_size} batches")
        
        # Check class distribution
        self._check_class_distribution()
        
        # Calculate class weights
        if self.config.USE_BALANCED_WEIGHTS:
            self._calculate_class_weights()
        
        return self.train_ds, self.val_ds, self.test_ds
    
    def _check_class_distribution(self):
        """Check and display class distribution in training set."""
        print("\n📊 Checking class distribution...")
        
        labels = []
        for _, y in self.train_ds:
            labels.extend(y.numpy().tolist())
        labels = np.array(labels)
        
        class_counts = np.bincount(labels.astype(int))
        total = len(labels)
        
        print("   Training set distribution:")
        for i, (name, count) in enumerate(zip(self.class_names, class_counts)):
            percentage = (count / total) * 100
            bar = '█' * int(percentage / 2) + '░' * (50 - int(percentage / 2))
            print(f"   {name:12} : {count:6} ({percentage:5.1f}%) [{bar}]")
        
        min_count = class_counts.min()
        max_count = class_counts.max()
        imbalance_ratio = max_count / min_count
        
        if imbalance_ratio > 2:
            print(f"\n   ⚠️  Class imbalance detected! Ratio: {imbalance_ratio:.2f}:1")
            print("   → Using class weights to prioritize minority classes")
        else:
            print(f"\n   ✅ Dataset is relatively balanced")
        
        return class_counts
    
    def _calculate_class_weights(self):
        """Calculate class weights for medical safety."""
        print("\n⚖️  Calculating class weights (Medical Safety Priority)...")
        
        labels = []
        for _, y in self.train_ds:
            labels.extend(y.numpy().tolist())
        labels = np.array(labels)
        
        classes = np.unique(labels)
        weights = class_weight.compute_class_weight(
            class_weight='balanced',
            classes=classes,
            y=labels
        )
        
        # Medical safety boost
        medical_boost = [1.0, 1.2, 1.5, 2.0]
        weights = weights * medical_boost[:len(classes)]
        weights = weights / weights.mean()
        
        self.class_weights = dict(enumerate(weights))
        
        print("   Class Weights (higher = more important):")
        for i, (name, weight) in enumerate(zip(self.class_names, weights)):
            print(f"   {name:12} : {weight:.3f}")
            
        print("\n   🏥 Clinical Rationale:")
        print("   → Grade 3 (Severe) gets highest weight")
        print("   → Grade 2 (Moderate) gets elevated weight")
        print("   → Grade 0 (Normal) gets lowest weight")
        print("   → This prioritizes Sensitivity (Recall) over Specificity")
        
        return self.class_weights

# Load data
loader = DataLoader(config)
train_ds, val_ds, test_ds = loader.load_data()
class_names = loader.class_names
num_classes = loader.num_classes
class_weights = loader.class_weights

## 🔄 Data Preparation & Augmentation

In [ ]:
def create_augmentation_pipeline(config):
    """Create data augmentation pipeline for training."""
    return models.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(config.AUGMENTATION['rotation']),
        layers.RandomZoom(config.AUGMENTATION['zoom']),
        layers.RandomContrast(config.AUGMENTATION['contrast']),
        layers.RandomBrightness(config.AUGMENTATION['brightness']),
    ])

def prepare_datasets(train_ds, val_ds, test_ds, config):
    """Prepare and optimize datasets for training."""
    AUTOTUNE = tf.data.AUTOTUNE
    augmentation = create_augmentation_pipeline(config)
    
    # Training set with augmentation
    train_ds = train_ds.repeat()
    train_ds = train_ds.map(
        lambda x, y: (augmentation(x, training=True), y),
        num_parallel_calls=AUTOTUNE
    )
    train_ds = train_ds.shuffle(1000, seed=config.RANDOM_SEED)
    train_ds = train_ds.batch(config.BATCH_SIZE)
    train_ds = train_ds.prefetch(AUTOTUNE)
    
    # Validation set
    val_ds = val_ds.repeat()
    val_ds = val_ds.batch(config.BATCH_SIZE)
    val_ds = val_ds.prefetch(AUTOTUNE)
    
    # Test set
    test_ds = test_ds.batch(config.BATCH_SIZE)
    test_ds = test_ds.prefetch(AUTOTUNE)
    
    # Calculate steps
    train_size = tf.data.experimental.cardinality(train_ds).numpy()
    val_size = tf.data.experimental.cardinality(val_ds).numpy()
    
    steps_per_epoch = train_size // 10
    validation_steps = val_size // 10
    
    print(f"\n📊 Training steps: {steps_per_epoch}, Validation steps: {validation_steps}")
    
    return train_ds, val_ds, test_ds, steps_per_epoch, validation_steps

# Prepare datasets
train_ds, val_ds, test_ds, steps_per_epoch, validation_steps = prepare_datasets(
    train_ds, val_ds, test_ds, config
)
config.train_steps = steps_per_epoch
config.val_steps = validation_steps

## 🔨 Build MobileNetV2 Model

In [ ]:
def build_model(num_classes, img_size, learning_rate=0.001):
    """Build the MobileNetV2-based model for RHD classification."""
    print("\n🔨 BUILDING MOBILENETV2 MODEL")
    print("="*60)
    print("Architecture: MobileNetV2 (Mobile-First CNN)")
    print("Rationale: Deployable in resource-limited African clinics")
    print("Transfer Learning: Pre-trained on ImageNet")
    
    base_model = tf.keras.applications.MobileNetV2(
        include_top=False,
        weights='imagenet',
        input_shape=(*img_size, 3)
    )
    base_model.trainable = False
    
    print(f"Base model: {base_model.name}")
    print(f"Total parameters: {base_model.count_params():,}")
    print("Initial state: Frozen (transfer learning)")
    
    model = models.Sequential([
        layers.Input(shape=(*img_size, 3)),
        layers.Rescaling(1./127.5, offset=-1),
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax')
    ])
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss='sparse_categorical_crossentropy',
        metrics=[
            'accuracy',
            tf.keras.metrics.SparseTopKCategoricalAccuracy(k=2, name='top_2_accuracy')
        ]
    )
    
    print("\n📊 Model Summary:")
    model.summary()
    
    return model, base_model

# Build model
model, base_model = build_model(num_classes, config.IMG_SIZE, config.INITIAL_LR)

## 📋 Training Callbacks

In [ ]:
def create_callbacks(config):
    """Create training callbacks for optimal learning."""
    callbacks = []
    
    checkpoint_path = os.path.join(
        config.OUTPUT_PATH, 'models',
        f"{config.RUN_NAME}_best.keras"
    )
    callbacks.append(
        tf.keras.callbacks.ModelCheckpoint(
            checkpoint_path,
            monitor='val_accuracy',
            save_best_only=True,
            mode='max',
            verbose=1
        )
    )
    
    callbacks.append(
        tf.keras.callbacks.EarlyStopping(
            monitor='val_accuracy',
            patience=7,
            restore_best_weights=True,
            mode='max',
            verbose=1
        )
    )
    
    callbacks.append(
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=3,
            min_lr=1e-7,
            verbose=1
        )
    )
    
    def cosine_decay(epoch):
        initial_lr = config.INITIAL_LR
        total_epochs = config.EPOCHS
        decay = 0.5 * (1 + np.cos(np.pi * epoch / total_epochs))
        return initial_lr * decay
    
    callbacks.append(
        tf.keras.callbacks.LearningRateScheduler(
            cosine_decay,
            verbose=0
        )
    )
    
    print("\n📋 Callbacks configured:")
    print(f"   ✓ Model Checkpoint (best val_accuracy)")
    print(f"   ✓ Early Stopping (patience=7)")
    print(f"   ✓ Reduce LR (factor=0.5, patience=3)")
    print(f"   ✓ Cosine Decay Learning Rate")
    
    return callbacks

# Create callbacks
callbacks = create_callbacks(config)

## 🚀 Training

In [ ]:
# Training parameters
print("\n" + "="*60)
print("🚀 STARTING TRAINING")
print("="*60)
print(f"Epochs: {config.EPOCHS}")
print(f"Steps per epoch: {steps_per_epoch}")
print(f"Class weights: {class_weights}")

# Train the model
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=config.EPOCHS,
    steps_per_epoch=steps_per_epoch,
    validation_steps=validation_steps,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1
)

## 📊 Evaluation

In [ ]:
print("\n📊 MODEL EVALUATION")
print("="*60)

# Basic evaluation
test_loss, test_acc, test_top2 = model.evaluate(test_ds, verbose=1)
print(f"\n📈 Test Results:")
print(f"   Loss: {test_loss:.4f}")
print(f"   Accuracy: {test_acc:.4f}")
print(f"   Top-2 Accuracy: {test_top2:.4f}")

# Predictions
print("\n📋 Generating predictions...")
y_true = []
y_pred = []

for images, labels in test_ds:
    predictions = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(predictions, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Classification report
print("\n📋 Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix - RHD Classification', fontsize=14)
plt.xlabel('Predicted Grade', fontsize=12)
plt.ylabel('True Grade', fontsize=12)
plt.tight_layout()
plt.show()

## 📈 Plot Training History

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy plot
axes[0].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy', linewidth=2)
axes[0].set_title('Model Accuracy', fontsize=14)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss plot
axes[1].plot(history.history['loss'], label='Train Loss', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Val Loss', linewidth=2)
axes[1].set_title('Model Loss', fontsize=14)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 💾 Export Models

In [ ]:
print("\n💾 EXPORTING MODELS")
print("="*60)

# 1. Keras format
keras_path = os.path.join(config.OUTPUT_PATH, 'models', f"{config.RUN_NAME}.keras")
model.save(keras_path)
print(f"✓ Keras model: {keras_path}")

# 2. TFLite format
try:
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    tflite_model = converter.convert()
    
    tflite_path = os.path.join(config.OUTPUT_PATH, 'models', f"{config.RUN_NAME}.tflite")
    with open(tflite_path, 'wb') as f:
        f.write(tflite_model)
    
    file_size = os.path.getsize(tflite_path) / (1024 * 1024)
    print(f"✓ TFLite model: {tflite_path} ({file_size:.2f} MB)")
except Exception as e:
    print(f"⚠️  TFLite export failed: {e}")

## 📊 Save Results

In [ ]:
# Save results
results = {
    'test_loss': float(test_loss),
    'test_accuracy': float(test_acc),
    'test_top2_accuracy': float(test_top2),
    'timestamp': config.TIMESTAMP,
    'model_name': config.RUN_NAME
}

results_path = os.path.join(config.OUTPUT_PATH, 'logs', f"{config.RUN_NAME}_results.json")
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f"✓ Results saved: {results_path}")

# Save history
history_path = os.path.join(config.OUTPUT_PATH, 'logs', f"{config.RUN_NAME}_history.pkl")
with open(history_path, 'wb') as f:
    pickle.dump(history.history, f)
print(f"✓ Training history saved: {history_path}")

## ✅ Complete!

**Summary:**
- Test Accuracy: {test_acc:.4f}
- Test Top-2 Accuracy: {test_top2:.4f}
- Model saved to: {config.OUTPUT_PATH}/models/